# HW14 — Эмбеддинги, FAISS, оценка retrieval и mini-RAG

**Цель:** построить учебный retrieval-конвейер — от базы знаний и чанкинга до индекса FAISS,
оценки качества поиска, обновления базы знаний и простого mini-RAG.

**Предметная область:** Солнечная система — планеты, спутники и другие объекты.

## 1. Импорты, seed и среда

In [1]:
import os
import random
import re
import textwrap
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print(f"Seed: {SEED}")
print(f"Устройство: {DEVICE}")

Seed: 42
Устройство: cpu


## 2. База знаний и первичный анализ

Используем собственную базу знаний о **Солнечной системе** — 20 документов, каждый
описывает планету, спутник или другой объект. Тема удобна для retrieval: есть много
фактов, по которым можно задавать вопросы.

In [2]:
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Меркурий",
        "text": (
            "Меркурий — ближайшая к Солнцу планета Солнечной системы. "
            "Его орбитальный период составляет около 88 земных суток. "
            "Меркурий не имеет атмосферы в обычном смысле: у него есть лишь крайне разрежённая экзосфера. "
            "Поверхность покрыта кратерами и напоминает лунную. Температура на дневной стороне "
            "достигает +430 °C, а на ночной опускается до −180 °C. "
            "Меркурий — самая маленькая планета Солнечной системы после исключения Плутона из списка планет."
        ),
    },
    {
        "doc_id": "doc_02",
        "title": "Венера",
        "text": (
            "Венера — вторая планета от Солнца. Она часто называется сестрой Земли из-за схожего размера. "
            "Однако условия на Венере крайне суровы: температура поверхности около 465 °C, "
            "атмосферное давление в 90 раз выше земного. Атмосфера состоит в основном из углекислого газа "
            "с облаками серной кислоты. Венера вращается вокруг своей оси в обратном направлении "
            "(ретроградное вращение) и делает один оборот за 243 земных суток — дольше, чем её год (225 суток)."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Земля",
        "text": (
            "Земля — третья планета от Солнца и единственная известная планета, на которой существует жизнь. "
            "Около 71% поверхности занимает вода. Земля имеет мощное магнитное поле, защищающее "
            "от солнечного ветра. Атмосфера состоит из 78% азота, 21% кислорода и примесей других газов. "
            "Средняя температура на поверхности составляет около 15 °C. У Земли один естественный спутник — Луна."
        ),
    },
    {
        "doc_id": "doc_04",
        "title": "Марс",
        "text": (
            "Марс — четвёртая планета от Солнца, известная как Красная планета из-за оксида железа на поверхности. "
            "Марс имеет тонкую атмосферу, состоящую в основном из углекислого газа. "
            "На Марсе находится самая высокая гора в Солнечной системе — Олимп (около 21 км). "
            "Марсианские сутки (сол) длятся примерно 24 часа 37 минут. "
            "У Марса два маленьких спутника — Фобос и Деймос, вероятно захваченные астероиды."
        ),
    },
    {
        "doc_id": "doc_05",
        "title": "Юпитер",
        "text": (
            "Юпитер — пятая планета от Солнца и крупнейшая в Солнечной системе. "
            "Масса Юпитера в 318 раз превышает массу Земли. Это газовый гигант, "
            "состоящий в основном из водорода и гелия. Большое Красное Пятно — гигантский "
            "ураган, бушующий на Юпитере уже несколько столетий. У Юпитера более 90 известных "
            "спутников, крупнейшие из которых — Ио, Европа, Ганимед и Каллисто (галилеевы спутники)."
        ),
    },
    {
        "doc_id": "doc_06",
        "title": "Сатурн",
        "text": (
            "Сатурн — шестая планета от Солнца, знаменитая своей системой колец. "
            "Кольца состоят из частиц льда и камня размером от пылинок до нескольких метров. "
            "Сатурн — газовый гигант, его плотность меньше плотности воды. "
            "У Сатурна более 140 известных спутников, крупнейший — Титан, "
            "который имеет плотную атмосферу из азота и метановые озёра на поверхности."
        ),
    },
    {
        "doc_id": "doc_07",
        "title": "Уран",
        "text": (
            "Уран — седьмая планета от Солнца, ледяной гигант. "
            "Уникальная особенность — ось вращения наклонена почти на 98°, поэтому планета "
            "как бы «лежит на боку». Атмосфера содержит водород, гелий и метан, "
            "который придаёт планете голубовато-зелёный цвет. "
            "Уран имеет систему тонких колец и 27 известных спутников, "
            "названных в честь персонажей Шекспира и Александра Поупа."
        ),
    },
    {
        "doc_id": "doc_08",
        "title": "Нептун",
        "text": (
            "Нептун — восьмая и самая удалённая планета Солнечной системы. "
            "Это ледяной гигант с сильнейшими ветрами в Солнечной системе — до 2100 км/ч. "
            "Атмосфера содержит водород, гелий и метан. Крупнейший спутник — Тритон, "
            "который движется по ретроградной орбите и, возможно, является захваченным "
            "объектом из пояса Койпера. У Нептуна 16 известных спутников и слабая система колец."
        ),
    },
    {
        "doc_id": "doc_09",
        "title": "Луна",
        "text": (
            "Луна — единственный естественный спутник Земли. Её диаметр составляет около 3474 км. "
            "Луна не имеет атмосферы и магнитного поля. Поверхность покрыта реголитом — "
            "мелкой пылью и обломками пород. На Луне побывали 12 человек в рамках программы «Аполлон» "
            "(1969–1972). Луна влияет на приливы и отливы на Земле и постепенно удаляется "
            "от неё примерно на 3,8 см в год."
        ),
    },
    {
        "doc_id": "doc_10",
        "title": "Солнце",
        "text": (
            "Солнце — звезда в центре Солнечной системы, жёлтый карлик спектрального класса G2V. "
            "Его масса составляет 99,86% от общей массы всей Солнечной системы. "
            "Температура поверхности (фотосферы) — около 5500 °C, а ядра — около 15 млн °C. "
            "Энергия вырабатывается за счёт термоядерного синтеза, в котором водород превращается в гелий. "
            "Возраст Солнца — около 4,6 млрд лет, оно находится примерно на середине своего жизненного цикла."
        ),
    },
    {
        "doc_id": "doc_11",
        "title": "Пояс астероидов",
        "text": (
            "Пояс астероидов — область между орбитами Марса и Юпитера, содержащая множество "
            "малых тел Солнечной системы. Крупнейший объект — карликовая планета Церера диаметром около 940 км. "
            "Суммарная масса всех объектов пояса составляет лишь около 4% массы Луны. "
            "Астероиды состоят из камня и металла. Пояс астероидов не является плотным скоплением — "
            "расстояния между объектами велики."
        ),
    },
    {
        "doc_id": "doc_12",
        "title": "Плутон",
        "text": (
            "Плутон — карликовая планета в поясе Койпера. До 2006 года считался девятой планетой. "
            "Диаметр Плутона — около 2377 км. У Плутона пять известных спутников, крупнейший — Харон, "
            "диаметром около 1212 км. Поверхность покрыта азотным льдом, а также льдами метана и CO. "
            "Миссия New Horizons в 2015 году впервые передала детальные снимки Плутона."
        ),
    },
    {
        "doc_id": "doc_13",
        "title": "Ио — спутник Юпитера",
        "text": (
            "Ио — один из четырёх галилеевых спутников Юпитера и самое вулканически активное "
            "тело в Солнечной системе. На Ио насчитывается более 400 действующих вулканов. "
            "Активность вызвана приливным нагревом от гравитационного взаимодействия с Юпитером, "
            "Европой и Ганимедом. Поверхность покрыта серой и диоксидом серы, что придаёт ей жёлто-оранжевый цвет."
        ),
    },
    {
        "doc_id": "doc_14",
        "title": "Европа — спутник Юпитера",
        "text": (
            "Европа — один из галилеевых спутников Юпитера. Поверхность покрыта гладким ледяным панцирем. "
            "Под ледяной коркой, вероятно, существует глобальный океан жидкой воды. "
            "Этот подлёдный океан делает Европу одним из главных кандидатов на поиск "
            "внеземной жизни в Солнечной системе. Толщина ледяного покрова оценивается в 10–30 км."
        ),
    },
    {
        "doc_id": "doc_15",
        "title": "Титан — спутник Сатурна",
        "text": (
            "Титан — крупнейший спутник Сатурна и второй по размеру спутник в Солнечной системе. "
            "Титан — единственный спутник с плотной атмосферой, состоящей в основном из азота. "
            "На его поверхности обнаружены озёра и моря из жидкого метана и этана. "
            "Температура на поверхности — около −179 °C. Миссия Кассини-Гюйгенс в 2005 году "
            "успешно высадила зонд Гюйгенс на поверхность Титана."
        ),
    },
    {
        "doc_id": "doc_16",
        "title": "Комета Галлея",
        "text": (
            "Комета Галлея — самая известная периодическая комета, возвращающаяся к Солнцу "
            "каждые 75–79 лет. Последний раз она наблюдалась в 1986 году, следующее появление "
            "ожидается в 2061 году. Ядро кометы имеет размер около 15×8 км. "
            "При приближении к Солнцу лёд испаряется, образуя кому и хвост длиной в миллионы километров."
        ),
    },
    {
        "doc_id": "doc_17",
        "title": "Пояс Койпера",
        "text": (
            "Пояс Койпера — область Солнечной системы за орбитой Нептуна, простирающаяся "
            "примерно от 30 до 55 а.е. от Солнца. Содержит множество ледяных тел, "
            "включая карликовые планеты Плутон, Эриду, Макемаке и Хаумеа. "
            "Объекты пояса Койпера состоят из замёрзших летучих веществ — метана, аммиака и воды. "
            "Пояс Койпера считается источником короткопериодических комет."
        ),
    },
    {
        "doc_id": "doc_18",
        "title": "Ганимед — спутник Юпитера",
        "text": (
            "Ганимед — крупнейший спутник в Солнечной системе, его диаметр составляет 5268 км, "
            "что больше Меркурия. Ганимед — единственный спутник, обладающий собственным магнитным полем. "
            "Поверхность состоит из двух типов рельефа: тёмные старые области с кратерами "
            "и светлые более молодые области с бороздами. Под поверхностью, возможно, есть солёный океан."
        ),
    },
    {
        "doc_id": "doc_19",
        "title": "Облако Оорта",
        "text": (
            "Облако Оорта — гипотетическая сферическая область на границе Солнечной системы, "
            "простирающаяся на расстояние от 2000 до 100 000 а.е. от Солнца. "
            "Считается источником долгопериодических комет. Облако содержит триллионы ледяных тел. "
            "Ни один космический аппарат пока не достиг облака Оорта — "
            "Вояджер-1, самый далёкий аппарат, всё ещё находится далеко от его внутренней границы."
        ),
    },
    {
        "doc_id": "doc_20",
        "title": "Тритон — спутник Нептуна",
        "text": (
            "Тритон — крупнейший спутник Нептуна с диаметром около 2707 км. "
            "Он движется по ретроградной орбите, что указывает на захват из пояса Койпера. "
            "Поверхность покрыта азотным льдом, температура составляет −235 °C — одна из самых низких "
            "в Солнечной системе. На Тритоне обнаружены криовулканы — гейзеры, "
            "выбрасывающие азот на высоту до 8 км."
        ),
    },
]

print(f"Количество документов: {len(documents)}")
print()
for doc in documents[:5]:
    print(f"[{doc['doc_id']}] {doc['title']}")
    print(textwrap.shorten(doc['text'], width=120, placeholder="..."))
    print()


Количество документов: 20

[doc_01] Меркурий
Меркурий — ближайшая к Солнцу планета Солнечной системы. Его орбитальный период составляет около 88 земных суток....

[doc_02] Венера
Венера — вторая планета от Солнца. Она часто называется сестрой Земли из-за схожего размера. Однако условия на Венере...

[doc_03] Земля
Земля — третья планета от Солнца и единственная известная планета, на которой существует жизнь. Около 71% поверхности...

[doc_04] Марс
Марс — четвёртая планета от Солнца, известная как Красная планета из-за оксида железа на поверхности. Марс имеет...

[doc_05] Юпитер
Юпитер — пятая планета от Солнца и крупнейшая в Солнечной системе. Масса Юпитера в 318 раз превышает массу Земли. Это...



## 3. Чанкинг документов

Разбиваем каждый документ на фрагменты по словам с перекрытием.
- `chunk_size=30` слов — достаточно, чтобы сохранить смысл.
- `overlap=10` слов — обеспечивает контекст на стыках.

In [3]:
def chunk_text(text: str, chunk_size: int = 30, overlap: int = 10) -> List[str]:
    """Разбивает текст на чанки по словам с перекрытием."""
    words = text.replace("\n", " ").split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_chunks(docs: List[Dict[str, str]], chunk_size: int = 30, overlap: int = 10) -> pd.DataFrame:
    """Чанкит все документы, возвращает DataFrame."""
    rows = []
    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for idx, chunk in enumerate(chunks):
            rows.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": idx,
                "chunk_text": chunk,
            })
    return pd.DataFrame(rows)


CHUNK_SIZE = 30
OVERLAP = 10

chunks_df = build_chunks(documents, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
print(f"Количество чанков: {len(chunks_df)}")
print()

# Пример чанкинга для первого документа
example_doc = documents[0]
example_chunks = chunk_text(example_doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
print(f"Документ: {example_doc['title']}")
print(f"Исходный текст ({len(example_doc['text'].split())} слов):")
print(example_doc["text"])
print(f"\nЧанки ({len(example_chunks)} шт.):")
for i, ch in enumerate(example_chunks):
    print(f"  Чанк {i}: {ch}")


Количество чанков: 55

Документ: Меркурий
Исходный текст (63 слов):
Меркурий — ближайшая к Солнцу планета Солнечной системы. Его орбитальный период составляет около 88 земных суток. Меркурий не имеет атмосферы в обычном смысле: у него есть лишь крайне разрежённая экзосфера. Поверхность покрыта кратерами и напоминает лунную. Температура на дневной стороне достигает +430 °C, а на ночной опускается до −180 °C. Меркурий — самая маленькая планета Солнечной системы после исключения Плутона из списка планет.

Чанки (3 шт.):
  Чанк 0: Меркурий — ближайшая к Солнцу планета Солнечной системы. Его орбитальный период составляет около 88 земных суток. Меркурий не имеет атмосферы в обычном смысле: у него есть лишь крайне разрежённая экзосфера.
  Чанк 1: в обычном смысле: у него есть лишь крайне разрежённая экзосфера. Поверхность покрыта кратерами и напоминает лунную. Температура на дневной стороне достигает +430 °C, а на ночной опускается до −180 °C.
  Чанк 2: достигает +430 °C, а на ночной опускает

## 4. Эмбеддинги и индекс FAISS

Используем модель `paraphrase-multilingual-MiniLM-L12-v2` из `sentence-transformers` —
мультиязычная, поддерживает русский, 384 измерения. Эмбеддинги нормализуем, строим `IndexFlatIP` для cosine-similarity.

In [4]:
# Загружаем модель
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)
print(f"Модель: paraphrase-multilingual-MiniLM-L12-v2, устройство: {DEVICE}")

# Получаем эмбеддинги для всех чанков
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_vectors = model.encode(chunk_texts, batch_size=32, show_progress_bar=False,
                              convert_to_numpy=True, normalize_embeddings=True).astype("float32")

print(f"Матрица эмбеддингов: {chunk_vectors.shape}")

# Строим индекс FAISS (IndexFlatIP для cosine similarity)
dim = chunk_vectors.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vectors)
print(f"Индекс FAISS создан: {index.ntotal} векторов, размерность {dim}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, устройство: cpu


Матрица эмбеддингов: (55, 384)
Индекс FAISS создан: 55 векторов, размерность 384


In [5]:
def search(query: str, model, index, chunks_df, top_k: int = 5) -> pd.DataFrame:
    """Поиск top-k ближайших чанков по запросу."""
    q_vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, indices = index.search(q_vec, top_k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        row = chunks_df.iloc[idx]
        results.append({
            "rank": rank + 1,
            "doc_id": row["doc_id"],
            "title": row["title"],
            "chunk_id": row["chunk_id"],
            "score": round(float(score), 4),
            "chunk_text": row["chunk_text"],
        })
    return pd.DataFrame(results)


# Демонстрация поиска по 5 запросам
demo_queries = [
    "Какая планета ближе всего к Солнцу?",
    "Где в Солнечной системе могут найти жизнь?",
    "Самая высокая гора в Солнечной системе",
    "Какие планеты имеют кольца?",
    "Когда вернётся комета Галлея?",
]

for q in demo_queries:
    print(f"\nЗапрос: {q}")
    res = search(q, model, index, chunks_df, top_k=3)
    for _, r in res.iterrows():
        print(f"  [{r['rank']}] {r['title']} (chunk {r['chunk_id']}, score={r['score']:.4f}): {r['chunk_text'][:80]}...")



Запрос: Какая планета ближе всего к Солнцу?
  [1] Нептун (chunk 0, score=0.6800): Нептун — восьмая и самая удалённая планета Солнечной системы. Это ледяной гигант...
  [2] Меркурий (chunk 0, score=0.6773): Меркурий — ближайшая к Солнцу планета Солнечной системы. Его орбитальный период ...
  [3] Пояс Койпера (chunk 0, score=0.5749): Пояс Койпера — область Солнечной системы за орбитой Нептуна, простирающаяся прим...

Запрос: Где в Солнечной системе могут найти жизнь?
  [1] Меркурий (chunk 0, score=0.5655): Меркурий — ближайшая к Солнцу планета Солнечной системы. Его орбитальный период ...
  [2] Пояс Койпера (chunk 0, score=0.5539): Пояс Койпера — область Солнечной системы за орбитой Нептуна, простирающаяся прим...
  [3] Нептун (chunk 1, score=0.5473): 2100 км/ч. Атмосфера содержит водород, гелий и метан. Крупнейший спутник — Трито...

Запрос: Самая высокая гора в Солнечной системе
  [1] Ганимед — спутник Юпитера (chunk 0, score=0.6134): Ганимед — крупнейший спутник в Солнечной системе, 

## 5. Контрольные запросы и оценка retrieval

Готовим 10 контрольных запросов с ожидаемыми релевантными документами.
Считаем `hit@k`, `recall@k` и `MRR@k`.

In [6]:
# Контрольные запросы: query -> список ожидаемых doc_id
eval_queries = [
    {"query": "Ближайшая планета к Солнцу", "expected": ["doc_01"]},
    {"query": "Температура на поверхности Венеры", "expected": ["doc_02"]},
    {"query": "Сколько воды на Земле", "expected": ["doc_03"]},
    {"query": "Самая высокая гора Олимп", "expected": ["doc_04"]},
    {"query": "Большое Красное Пятно Юпитера", "expected": ["doc_05"]},
    {"query": "Кольца Сатурна из чего состоят", "expected": ["doc_06"]},
    {"query": "Спутники названные в честь Шекспира", "expected": ["doc_07"]},
    {"query": "Сильнейшие ветры на Нептуне", "expected": ["doc_08"]},
    {"query": "Программа Аполлон Луна", "expected": ["doc_09"]},
    {"query": "Подлёдный океан на Европе", "expected": ["doc_14"]},
]

TOP_K = 5

def evaluate_retrieval(queries, model, index, chunks_df, top_k=5):
    """Оценка retrieval: hit@k, recall@k, MRR@k."""
    results = []
    hits = 0
    total_recall = 0.0
    total_mrr = 0.0

    for item in queries:
        q = item["query"]
        expected = set(item["expected"])
        res = search(q, model, index, chunks_df, top_k=top_k)
        retrieved_docs = res["doc_id"].tolist()
        retrieved_unique = list(dict.fromkeys(retrieved_docs))  # уникальные с порядком

        # hit@k — хотя бы один ожидаемый документ в top-k
        hit = int(any(d in expected for d in retrieved_unique))
        hits += hit

        # recall@k — доля найденных ожидаемых
        found = sum(1 for d in expected if d in retrieved_unique)
        recall = found / len(expected)
        total_recall += recall

        # MRR@k — 1 / позиция первого релевантного
        rank_first = 0
        for i, d in enumerate(retrieved_unique):
            if d in expected:
                rank_first = i + 1
                break
        mrr = (1.0 / rank_first) if rank_first > 0 else 0.0
        total_mrr += mrr

        results.append({
            "query": q,
            "expected_source": ", ".join(expected),
            "retrieved_sources": ", ".join(retrieved_unique[:top_k]),
            "hit_at_k": hit,
            "rank_of_first_relevant": rank_first if rank_first > 0 else None,
        })

    n = len(queries)
    metrics = {
        "hit@k": hits / n,
        "recall@k": total_recall / n,
        "MRR@k": total_mrr / n,
    }
    return pd.DataFrame(results), metrics


eval_df, metrics = evaluate_retrieval(eval_queries, model, index, chunks_df, top_k=TOP_K)
print(f"Результаты оценки retrieval (top_k={TOP_K}):")
print(f"  hit@{TOP_K}    = {metrics['hit@k']:.2f}")
print(f"  recall@{TOP_K} = {metrics['recall@k']:.2f}")
print(f"  MRR@{TOP_K}   = {metrics['MRR@k']:.2f}")
print()
display(eval_df)

# Сохраняем артефакт
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
eval_df.to_csv(os.path.join(ARTIFACTS_DIR, "retrieval_eval.csv"), index=False)
print(f"\nСохранено: artifacts/retrieval_eval.csv")


Результаты оценки retrieval (top_k=5):
  hit@5    = 1.00
  recall@5 = 1.00
  MRR@5   = 0.95



,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,Ближайшая планета к Солнцу,doc_01,"doc_08, doc_01, doc_10, doc_12",1,2
1,Температура на поверхности Венеры,doc_02,"doc_02, doc_03, doc_01, doc_10",1,1
2,Сколько воды на Земле,doc_03,"doc_03, doc_06, doc_15, doc_09",1,1
3,Самая высокая гора Олимп,doc_04,"doc_04, doc_18, doc_11, doc_08",1,1
4,Большое Красное Пятно Юпитера,doc_05,"doc_05, doc_11, doc_08",1,1
5,Кольца Сатурна из чего состоят,doc_06,"doc_06, doc_17, doc_11, doc_20",1,1
6,Спутники названные в честь Шекспира,doc_07,"doc_07, doc_05, doc_11, doc_09",1,1
7,Сильнейшие ветры на Нептуне,doc_08,"doc_08, doc_07, doc_13, doc_19, doc_02",1,1
8,Программа Аполлон Луна,doc_09,"doc_09, doc_08, doc_03, doc_17",1,1
9,Подлёдный океан на Европе,doc_14,"doc_14, doc_15, doc_17, doc_13",1,1



Сохранено: artifacts/retrieval_eval.csv


## 6. Эксперимент с параметрами retrieval

Сравниваем два значения `chunk_size`: 30 (текущий) и 50 слов.
Цель — увидеть, как размер чанка влияет на качество retrieval.

In [7]:
# Эксперимент: chunk_size=30 vs chunk_size=50
CHUNK_SIZE_ALT = 50
OVERLAP_ALT = 15

chunks_df_alt = build_chunks(documents, chunk_size=CHUNK_SIZE_ALT, overlap=OVERLAP_ALT)
print(f"chunk_size={CHUNK_SIZE_ALT}, overlap={OVERLAP_ALT}: {len(chunks_df_alt)} чанков")

chunk_texts_alt = chunks_df_alt["chunk_text"].tolist()
chunk_vectors_alt = model.encode(chunk_texts_alt, batch_size=32, show_progress_bar=False,
                                  convert_to_numpy=True, normalize_embeddings=True).astype("float32")
index_alt = faiss.IndexFlatIP(chunk_vectors_alt.shape[1])
index_alt.add(chunk_vectors_alt)

eval_df_alt, metrics_alt = evaluate_retrieval(eval_queries, model, index_alt, chunks_df_alt, top_k=TOP_K)

print(f"\nСравнение:")
print(f"  chunk_size={CHUNK_SIZE}:  hit@{TOP_K}={metrics['hit@k']:.2f}, recall@{TOP_K}={metrics['recall@k']:.2f}, MRR@{TOP_K}={metrics['MRR@k']:.2f}")
print(f"  chunk_size={CHUNK_SIZE_ALT}: hit@{TOP_K}={metrics_alt['hit@k']:.2f}, recall@{TOP_K}={metrics_alt['recall@k']:.2f}, MRR@{TOP_K}={metrics_alt['MRR@k']:.2f}")

print(f"\nВывод: ", end="")
if metrics['hit@k'] >= metrics_alt['hit@k']:
    print(f"chunk_size={CHUNK_SIZE} работает не хуже — оставляем как основной вариант.")
else:
    print(f"chunk_size={CHUNK_SIZE_ALT} показывает лучший результат.")
print("Меньший чанк даёт более точное сопоставление запроса с фрагментом,")
print("но увеличивает количество чанков и может разрывать контекст.")


chunk_size=50, overlap=15: 35 чанков



Сравнение:
  chunk_size=30:  hit@5=1.00, recall@5=1.00, MRR@5=0.95
  chunk_size=50: hit@5=1.00, recall@5=1.00, MRR@5=0.87

Вывод: chunk_size=30 работает не хуже — оставляем как основной вариант.
Меньший чанк даёт более точное сопоставление запроса с фрагментом,
но увеличивает количество чанков и может разрывать контекст.


## 7. Обновление базы знаний и переиндексация

Добавляем 3 новых документа. Показываем, как retrieval меняется до и после обновления.

In [8]:
# Новые документы
new_documents = [
    {
        "doc_id": "doc_21",
        "title": "Энцелад — спутник Сатурна",
        "text": (
            "Энцелад — шестой по размеру спутник Сатурна с диаметром около 500 км. "
            "Он привлёк внимание учёных благодаря гейзерам на южном полюсе, "
            "выбрасывающим струи водяного пара и ледяных частиц. "
            "Под ледяной поверхностью Энцелада находится глобальный океан жидкой воды. "
            "Наличие океана, энергии и органических молекул делает Энцелад "
            "одним из приоритетных объектов для поиска внеземной жизни."
        ),
    },
    {
        "doc_id": "doc_22",
        "title": "Церера — карликовая планета",
        "text": (
            "Церера — крупнейший объект в поясе астероидов и единственная карликовая планета "
            "во внутренней части Солнечной системы. Диаметр — около 940 км. "
            "Миссия Dawn обнаружила на поверхности яркие пятна, состоящие из солей натрия. "
            "Предполагается наличие подповерхностного резервуара солёной воды. "
            "Церера содержит значительное количество водяного льда."
        ),
    },
    {
        "doc_id": "doc_23",
        "title": "Вояджеры — межпланетные зонды",
        "text": (
            "Вояджер-1 и Вояджер-2 — космические аппараты NASA, запущенные в 1977 году. "
            "Вояджер-1 — самый далёкий от Земли рукотворный объект, вышедший в межзвёздное пространство в 2012 году. "
            "Вояджер-2 — единственный аппарат, пролетевший мимо Урана и Нептуна. "
            "На борту каждого Вояджера установлена «Золотая пластинка» с посланием внеземным цивилизациям. "
            "Оба аппарата продолжают передавать данные, хотя их мощность постепенно снижается."
        ),
    },
]

# Запросы, которые должны измениться после обновления
update_queries = [
    "Гейзеры и океан на спутнике Сатурна",
    "Карликовая планета в поясе астероидов",
    "Самый далёкий космический аппарат",
    "Поиск жизни на спутниках",
    "Кто пролетел мимо Урана",
]

# Retrieval ДО обновления
before_results = {}
for q in update_queries:
    res = search(q, model, index, chunks_df, top_k=TOP_K)
    before_results[q] = list(dict.fromkeys(res["doc_id"].tolist()))

# Обновляем базу знаний
documents_updated = documents + new_documents
chunks_df_updated = build_chunks(documents_updated, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
print(f"Обновлённая база: {len(documents_updated)} документов, {len(chunks_df_updated)} чанков")

# Пересчитываем эмбеддинги и переиндексируем
chunk_texts_updated = chunks_df_updated["chunk_text"].tolist()
chunk_vectors_updated = model.encode(chunk_texts_updated, batch_size=32, show_progress_bar=False,
                                      convert_to_numpy=True, normalize_embeddings=True).astype("float32")
index_updated = faiss.IndexFlatIP(chunk_vectors_updated.shape[1])
index_updated.add(chunk_vectors_updated)
print(f"Индекс обновлён: {index_updated.ntotal} векторов")

# Retrieval ПОСЛЕ обновления
after_results = {}
for q in update_queries:
    res = search(q, model, index_updated, chunks_df_updated, top_k=TOP_K)
    after_results[q] = list(dict.fromkeys(res["doc_id"].tolist()))

# Сравнение
comparison_rows = []
for q in update_queries:
    before = before_results[q]
    after = after_results[q]
    changed = before != after
    comparison_rows.append({
        "query": q,
        "before_retrieved_sources": ", ".join(before),
        "after_retrieved_sources": ", ".join(after),
        "changed": changed,
    })
    print(f"\nЗапрос: {q}")
    print(f"  До:    {before}")
    print(f"  После: {after}")
    print(f"  Изменилось: {'Да' if changed else 'Нет'}")

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(os.path.join(ARTIFACTS_DIR, "retrieval_before_after_update.csv"), index=False)
print(f"\nСохранено: artifacts/retrieval_before_after_update.csv")


Обновлённая база: 23 документов, 63 чанков


Индекс обновлён: 63 векторов

Запрос: Гейзеры и океан на спутнике Сатурна
  До:    ['doc_06', 'doc_17', 'doc_14', 'doc_08']
  После: ['doc_21', 'doc_06', 'doc_17', 'doc_14']
  Изменилось: Да

Запрос: Карликовая планета в поясе астероидов
  До:    ['doc_11', 'doc_17', 'doc_12']
  После: ['doc_11', 'doc_22', 'doc_17', 'doc_12']
  Изменилось: Да

Запрос: Самый далёкий космический аппарат
  До:    ['doc_19', 'doc_08']
  После: ['doc_19', 'doc_23', 'doc_22', 'doc_08']
  Изменилось: Да

Запрос: Поиск жизни на спутниках
  До:    ['doc_14', 'doc_04', 'doc_07', 'doc_19', 'doc_09']
  После: ['doc_14', 'doc_21', 'doc_04', 'doc_07', 'doc_23']
  Изменилось: Да

Запрос: Кто пролетел мимо Урана
  До:    ['doc_07', 'doc_08', 'doc_12']
  После: ['doc_23', 'doc_07', 'doc_08', 'doc_22']
  Изменилось: Да

Сохранено: artifacts/retrieval_before_after_update.csv


## 8. Mini-RAG

Простой RAG-конвейер:
1. Получаем запрос.
2. Ищем top-k релевантных чанков через FAISS.
3. Формируем контекст из найденных фрагментов.
4. Генерируем ответ шаблонным генератором (на основе ключевых слов и контекста).
5. Возвращаем ответ с указанием источников.

Используем шаблонный генератор без внешних LLM-сервисов.

In [9]:
def template_answer(query: str, context_chunks: List[Dict]) -> str:
    """Шаблонный генератор ответа на основе найденного контекста.

    Объединяет найденные фрагменты в связный ответ без использования LLM.
    """
    if not context_chunks:
        return "К сожалению, в базе знаний не найдено релевантной информации для ответа на этот вопрос."

    # Собираем контекст
    context_parts = []
    for chunk in context_chunks:
        context_parts.append(chunk["chunk_text"])

    full_context = " ".join(context_parts)

    # Извлекаем предложения, наиболее релевантные запросу
    sentences = re.split(r'(?<=[.!?])\s+', full_context)
    query_words = set(query.lower().split())

    # Считаем релевантность каждого предложения
    scored = []
    for sent in sentences:
        sent_words = set(sent.lower().split())
        overlap = len(query_words & sent_words)
        scored.append((overlap, sent))

    scored.sort(key=lambda x: -x[0])

    # Берём до 5 наиболее релевантных предложений
    top_sentences = [s for _, s in scored[:5] if s.strip()]

    if not top_sentences:
        top_sentences = sentences[:3]

    answer = " ".join(top_sentences)
    return answer


def mini_rag(query: str, model, index, chunks_df, top_k: int = 3) -> Dict:
    """Mini-RAG: запрос -> retrieval -> ответ с источниками."""
    # Retrieval
    res = search(query, model, index, chunks_df, top_k=top_k)

    context_chunks = []
    sources = []
    for _, row in res.iterrows():
        context_chunks.append({
            "doc_id": row["doc_id"],
            "title": row["title"],
            "chunk_text": row["chunk_text"],
            "score": row["score"],
        })
        source_str = f"{row['title']} ({row['doc_id']})"
        if source_str not in sources:
            sources.append(source_str)

    # Генерация ответа
    answer = template_answer(query, context_chunks)

    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": "; ".join(sources),
        "context_chunks": context_chunks,
    }


# Демонстрация mini-RAG
rag_questions = [
    "Какая планета самая большая в Солнечной системе?",
    "Где могут обнаружить внеземную жизнь?",
    "Что находится на поверхности Титана?",
    "Какая температура на Меркурии?",
    "Откуда берутся кометы?",
]

print("=== Примеры работы Mini-RAG ===\n")
rag_results = []
for q in rag_questions:
    result = mini_rag(q, model, index_updated, chunks_df_updated, top_k=3)
    rag_results.append(result)
    print(f"Вопрос: {q}")
    print(f"Ответ: {result['answer'][:200]}...")
    print(f"Источники: {result['retrieved_sources']}")
    print()


=== Примеры работы Mini-RAG ===

Вопрос: Какая планета самая большая в Солнечной системе?
Ответ: Юпитер — пятая планета от Солнца и крупнейшая в Солнечной системе. Большое Нептун — восьмая и самая удалённая планета Солнечной системы. Это ледяной гигант с сильнейшими ветрами в Солнечной системе — ...
Источники: Юпитер (doc_05); Нептун (doc_08); Ганимед — спутник Юпитера (doc_18)

Вопрос: Где могут обнаружить внеземную жизнь?
Ответ: и органических молекул делает Энцелад одним из приоритетных объектов для поиска внеземной жизни. воды. Этот подлёдный океан делает Европу одним из главных кандидатов на поиск внеземной жизни в Солнечн...
Источники: Энцелад — спутник Сатурна (doc_21); Европа — спутник Юпитера (doc_14); Церера — карликовая планета (doc_22)

Вопрос: Что находится на поверхности Титана?
Ответ: Миссия Dawn обнаружила на поверхности яркие пятна, состоящие из Сатурн — шестая планета от Солнца, знаменитая своей системой колец. Церера — крупнейший объект в поясе астероидов и единствен

## 9. Краткий анализ ошибок

Проверяем mini-RAG на дополнительных запросах, включая сложные и пограничные случаи.

In [10]:
# Дополнительные запросы — включая пограничные и сложные
error_analysis_questions = [
    # Хорошие запросы
    "Сколько спутников у Юпитера?",
    "Какова масса Солнца?",
    "Что такое пояс Койпера?",
    # Пограничные / проблемные запросы
    "Какая погода на Марсе сегодня?",          # Нет данных о текущей погоде
    "Сравни гравитацию на Земле и на Луне",    # Нет данных о гравитации Луны в базе
    "Кто открыл Плутон?",                       # Нет информации об открытии в нашей базе
    "Какие экзопланеты похожи на Землю?",       # Вне области базы знаний
]

print("=== Анализ ошибок Mini-RAG ===\n")
all_rag_results = list(rag_results)  # Уже собранные результаты

for q in error_analysis_questions:
    result = mini_rag(q, model, index_updated, chunks_df_updated, top_k=3)
    all_rag_results.append(result)
    print(f"Вопрос: {q}")
    print(f"Ответ: {result['answer'][:200]}")
    print(f"Источники: {result['retrieved_sources']}")
    print()

# Комментарии к неудачным случаям
print("=" * 60)
print("КОММЕНТАРИИ К ПОГРАНИЧНЫМ СЛУЧАЯМ:")
print("=" * 60)
print()
print("1. «Какая погода на Марсе сегодня?»")
print("   Проблема: база знаний содержит общую информацию о Марсе,")
print("   но не имеет данных о текущей погоде. Retrieval находит")
print("   документ о Марсе, но ответ не соответствует вопросу.")
print()
print("2. «Сравни гравитацию на Земле и на Луне»")
print("   Проблема: база содержит информацию о Земле и Луне отдельно,")
print("   но нет данных о гравитации Луны. Retrieval корректно находит")
print("   оба документа, но данных для сравнения недостаточно.")
print()
print("3. «Кто открыл Плутон?»")
print("   Проблема: документ о Плутоне не содержит информации об")
print("   открытии (автор, год). Retrieval находит документ о Плутоне,")
print("   но ответ на вопрос отсутствует в контексте.")
print()
print("4. «Какие экзопланеты похожи на Землю?»")
print("   Проблема: экзопланеты полностью вне области базы знаний.")
print("   Retrieval находит документы о Земле и других планетах,")
print("   но они не содержат информации об экзопланетах.")


=== Анализ ошибок Mini-RAG ===

Вопрос: Сколько спутников у Юпитера?
Ответ: У Юпитера более 90 известных спутников, крупнейшие из которых — Ио, Европа, Ганимед и Каллисто (галилеевы спутники). У Плутона пять известных спутников, крупнейший — Харон, диаметром около 1212 столет
Источники: Юпитер (doc_05); Плутон (doc_12)

Вопрос: Какова масса Солнца?
Ответ: Его масса составляет 99,86% от общей массы всей Солнечной системы. Масса Юпитера в 318 раз превышает массу Земли. Солнце — звезда в центре Солнечной системы, жёлтый карлик спектрального класса G2V. Те
Источники: Солнце (doc_10); Ганимед — спутник Юпитера (doc_18); Юпитер (doc_05)



Вопрос: Что такое пояс Койпера?
Ответ: Пояс Койпера считается источником короткопериодических комет. Церера — крупнейший объект в поясе астероидов и единственная карликовая планета во внутренней части Солнечной системы. Диаметр — около 940
Источники: Церера — карликовая планета (doc_22); Нептун (doc_08); Пояс Койпера (doc_17)

Вопрос: Какая погода на Марсе сегодня?
Ответ: На Марсе находится самая высокая гора в Солнечной системе — Олимп (около 21 км). На Марсе находится самая высокая в обычном смысле: у него есть лишь крайне разрежённая экзосфера. У Марса Марс — четвёр
Источники: Марс (doc_04); Меркурий (doc_01)

Вопрос: Сравни гравитацию на Земле и на Луне
Ответ: На Луне покрыта реголитом — мелкой пылью и обломками пород. Луна влияет на приливы и отливы на Земле и постепенно удаляется от Церера диаметром около 940 км. На Луне побывали 12 человек в рамках прогр
Источники: Луна (doc_09); Пояс астероидов (doc_11)

Вопрос: Кто открыл Плутон?
Ответ: Плутон — карликовая планета в поясе Койп

## Сохранение артефактов

In [11]:
# Сохраняем rag_examples.csv
rag_rows = []
for r in all_rag_results:
    rag_rows.append({
        "question": r["question"],
        "answer": r["answer"],
        "retrieved_sources": r["retrieved_sources"],
    })

rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv(os.path.join(ARTIFACTS_DIR, "rag_examples.csv"), index=False)
print(f"Сохранено: artifacts/rag_examples.csv ({len(rag_df)} примеров)")

# Проверяем все артефакты
for fname in ["retrieval_eval.csv", "rag_examples.csv", "retrieval_before_after_update.csv"]:
    path = os.path.join(ARTIFACTS_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  {fname}: {len(df)} строк, столбцы: {list(df.columns)}")
    else:
        print(f"  {fname}: НЕ НАЙДЕН!")

print("\nВсе артефакты сохранены успешно.")


Сохранено: artifacts/rag_examples.csv (12 примеров)
  retrieval_eval.csv: 10 строк, столбцы: ['query', 'expected_source', 'retrieved_sources', 'hit_at_k', 'rank_of_first_relevant']


  rag_examples.csv: 12 строк, столбцы: ['question', 'answer', 'retrieved_sources']
  retrieval_before_after_update.csv: 5 строк, столбцы: ['query', 'before_retrieved_sources', 'after_retrieved_sources', 'changed']

Все артефакты сохранены успешно.
